In [2]:
!pip install transformers gradio torch

In [3]:
from transformers import pipeline

# Load a small but powerful text generation model
generator = pipeline("text-generation", model="gpt2", max_new_tokens=200, temperature=0.8)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [4]:
def generate_story(prompt):
    result = generator(prompt)
    return result[0]['generated_text']

def continue_story(excerpt):
    result = generator(excerpt + "\n\nWhat happened next...\n")
    return result[0]['generated_text']

def create_poem(topic):
    prompt = f"Write a short poem about {topic}. Rhyming, emotional:\n"
    result = generator(prompt)
    return result[0]['generated_text']

def generate_fantasy_character():
    prompt = "Create a fantasy character: name, race, appearance, personality, and one secret:\n"
    result = generator(prompt)
    return result[0]['generated_text']

def rewrite_tone(text, tone):
    tone_prompts = {
        "funny": f"Rewrite this in a hilarious, joking way:\n{text}",
        "formal": f"Rewrite this in very formal, academic English:\n{text}",
        "scary": f"Rewrite this as a creepy, horror version:\n{text}",
        "poetic": f"Rewrite this like a beautiful poem:\n{text}",
        "pirate": f"Rewrite this as a pirate speaking, with 'arrr' and 'matey':\n{text}"
    }
    prompt = tone_prompts.get(tone, f"Rewrite this:\n{text}")
    result = generator(prompt)
    return result[0]['generated_text']

def create_dialogue(character_a, character_b, topic):
    prompt = f"""Write a short dialogue between {character_a} and {character_b} about {topic}.
{character_a}:
{character_b}:
{character_a}:
{character_b}:"""
    result = generator(prompt)
    return result[0]['generated_text']

In [5]:
import gradio as gr

def app_function(choice, input_text, tone, char_a, char_b, topic):
    if choice == "Generate story":
        return generate_story(input_text)
    elif choice == "Continue story":
        return continue_story(input_text)
    elif choice == "Create poem":
        return create_poem(input_text)
    elif choice == "Generate fantasy character":
        return generate_fantasy_character()
    elif choice == "Rewrite text in different tone":
        return rewrite_tone(input_text, tone)
    elif choice == "Create dialogue between characters":
        return create_dialogue(char_a, char_b, topic)
    else:
        return "Please select a valid option."

with gr.Blocks(title="Syntax & Sorcery") as demo:
    gr.Markdown("## 🧙 Syntax & Sorcery: AI Creative Writing Workshop")
    gr.Markdown("Generate stories, continue unfinished tales, write poems, create fantasy characters, rewrite in any tone, or craft dialogues.")

    with gr.Row():
        with gr.Column():
            choice = gr.Radio(
                ["Generate story", "Continue story", "Create poem",
                 "Generate fantasy character", "Rewrite text in different tone",
                 "Create dialogue between characters"],
                label="What would you like to do?"
            )
            input_text = gr.Textbox(label="Your prompt / story excerpt / poem topic", lines=3, placeholder="e.g., A dragon who loves baking...")
            tone = gr.Dropdown(["funny", "formal", "scary", "poetic", "pirate"], label="Choose tone (for rewrite only)", visible=False)
            char_a = gr.Textbox(label="Character A name", placeholder="e.g., Gandalf", visible=False)
            char_b = gr.Textbox(label="Character B name", placeholder="e.g., Dobby", visible=False)
            topic_dialogue = gr.Textbox(label="Conversation topic", placeholder="e.g., the lost ring", visible=False)

            def update_visibility(choice):
                show_rewrite = (choice == "Rewrite text in different tone")
                show_dialogue = (choice == "Create dialogue between characters")
                return gr.update(visible=show_rewrite), gr.update(visible=show_dialogue), gr.update(visible=show_dialogue), gr.update(visible=show_dialogue)

            choice.change(update_visibility, choice, [tone, char_a, char_b, topic_dialogue])

        with gr.Column():
            output = gr.Textbox(label="✨ Your created content", lines=15)
            submit_btn = gr.Button("Cast Spell ✨")

    submit_btn.click(app_function, [choice, input_text, tone, char_a, char_b, topic_dialogue], output)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6e9210546f8e185036.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
